# Konnon Drawing Model: Free Kaggle GPU Training

This notebook trains the **Konnon Staged Transformer** vector drawing model using Kaggle's free GPU compute (NVIDIA Tesla T4 or P100).

### Instructions:
1. Under **Notebook options** on the right, set **Accelerator** to **GPU T4 x2** (or **GPU P100**).
2. Ensure **Internet** is turned **ON** in the settings.
3. Run all cells sequentially.

In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: GPU is not active. Please select GPU T4 x2 or P100 under Notebook Settings.")

In [ ]:
# 2. Clone Repository & Install Dependencies
import os
if not os.path.exists("konnon-drawing-model"):
    # Clone your repo or pull from GitHub:
    # !git clone https://github.com/your-username/konnon-drawing-model.git
    # Alternatively, if you uploaded the project files as a Kaggle Dataset, copy them here:
    !cp -r /kaggle/input/konnon-drawing-model ./konnon-drawing-model || true

%cd konnon-drawing-model
!pip install -q pillow numpy pyyaml psutil

In [ ]:
# 3. Generate Scaled Gym Training Dataset (10,500 Transitions)
import os
data_path = "data/processed/staged_gym_10k/train.jsonl"
if not os.path.exists(data_path):
    print("Generating scaled synthetic gym dataset (3,500 drawings -> 10,500 staged transitions)...")
    !python -m konnon_draw.gym.generator --num-drawings 3500 --out-dir data/processed/staged_gym_10k
else:
    print(f"Dataset already present at: {data_path}")

In [ ]:
# 4. Launch High-Speed GPU Training with CUDA & FP16 Mixed Precision
!python scripts/train_scaled_gym.py \
    --ablation full \
    --device cuda \
    --fp16 \
    --d_model 256 \
    --n_layers 6 \
    --n_heads 8 \
    --d_ff 1024 \
    --epochs 30 \
    --batch_size 64 \
    --lr 0.001

In [ ]:
# 5. Run Evaluation & Generate Contact Sheet
!python scripts/eval_real_staged.py \
    --checkpoint outputs_v2/scaled/full/checkpoint_best.pt \
    --num-samples 64 \
    --out-dir /kaggle/working/eval_results

# Display evaluation contact sheet
from IPython.display import Image, display
if os.path.exists("/kaggle/working/eval_results/real_eval_contact_sheet.png"):
    display(Image("/kaggle/working/eval_results/real_eval_contact_sheet.png"))

In [ ]:
# 6. Copy Best Checkpoint to Kaggle Output for Easy Download
import shutil
shutil.copy("outputs_v2/scaled/full/checkpoint_best.pt", "/kaggle/working/checkpoint_best.pt")
print("===============================================================")
print("  TRAINING COMPLETE!")
print("  Download 'checkpoint_best.pt' from the Kaggle Output panel.")
print("===============================================================")